OIEE

In [6]:
PATH_LOCAL_H5AD       = 'nsclc_scrnaseq_processado.h5ad'
PATH_GSE131907_ANNOT  = '/home/lait/teinxin/datasets/GSE131907_Lung_Cancer_cell_annotation.txt.gz'  
PATH_GSE131907_MATRIX = '/home/lait/teinxin/teinxin_tc/GSE131907/GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz'  
GSE189357_DIR         = '/home/lait/teinxin/datasets/sc357' # botar um s tem que ser fiel  tem que ta fiel ao caminho 357
OUTPUT_H5AD           = 'nsclc_integrado_3datasets.h5ad'
OUTPUT_DIR            = 'figuras_integracao'

N_HVG       = 3000
N_PCS       = 50
N_PCS_USE   = 30
N_NEIGHBORS = 15
LEIDEN_RES  = 0.5

In [7]:
import os, gzip, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import harmonypy as hm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sc.settings.verbosity = 2
np.random.seed(42)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 60)
print('INTEGRAÇÃO scRNA-seq NSCLC — 3 DATASETS')
print('=' * 60)
print(f'scanpy:    {sc.__version__}')
print(f'harmonypy: {hm.__version__}')

INTEGRAÇÃO scRNA-seq NSCLC — 3 DATASETS
scanpy:    1.11.5
harmonypy: 0.2.0


DATASET LOCAL

In [8]:
print('Carregando dataset local...')

adata_local = sc.read_h5ad(PATH_LOCAL_H5AD)
adata_local_raw = adata_local.raw.to_adata()
adata_local_raw.obs['leiden']    = adata_local.obs['leiden']
adata_local_raw.obs['cell_type'] = adata_local.obs['cell_type']
adata_local_raw.obs['batch']     = 'local_10x_5GEX'
adata_local_raw.obs['dataset']   = 'local_10x_5GEX'
del adata_local

print(f'  {adata_local_raw.n_obs:,} células × {adata_local_raw.n_vars:,} genes')
print('  Tipos celulares:')
for ct, n in adata_local_raw.obs['cell_type'].value_counts().items():
    print(f'    {ct}: {n}')

Carregando dataset local...
  5,988 células × 19,925 genes
  Tipos celulares:
    Celula B: 2189
    NK: 1458
    Monocito/Macrofago: 553
    Tumoral (PD-L1+): 540
    T CD8 Citotoxico: 438
    Desconhecido: 279
    Celula Plasmatica: 225
    Mastocito: 157
    T Regulatorio: 114
    Fibroblasto: 26
    Alveolar Tipo II: 9


2. GSE131907 — Kim et al. 2020

In [9]:
print('Carregando GSE131907 (Kim et al. 2020)...')

annot = pd.read_csv(PATH_GSE131907_ANNOT, sep='\t', index_col=0)
amostras_tumor = annot[annot['Sample'].str.startswith(('LUNG_T', 'LN_'))].index
celulas_alvo   = set(amostras_tumor)
print(f'  Células tumorais selecionadas: {len(amostras_tumor):,}')

print('  Lendo cabeçalho da matriz...')
with gzip.open(PATH_GSE131907_MATRIX, 'rt') as f:
    barcodes_matriz = f.readline().strip().split('\t')[1:]

idx_cols    = [i for i, b in enumerate(barcodes_matriz) if b in celulas_alvo]
barcodes_sel = [barcodes_matriz[i] for i in idx_cols]
print(f'  Carregando {len(idx_cols):,} células (pode levar ~5 min)...')

genes, rows_list, cols_list, data_list = [], [], [], []
with gzip.open(PATH_GSE131907_MATRIX, 'rt') as f:
    f.readline()
    for gene_idx, linha in enumerate(f):
        partes = linha.rstrip('\n').split('\t')
        genes.append(partes[0])
        vals = np.array([float(partes[j + 1]) for j in idx_cols], dtype=np.float32)
        nz   = np.nonzero(vals)[0]
        rows_list.extend([gene_idx] * len(nz))
        cols_list.extend(nz.tolist())
        data_list.extend(vals[nz].tolist())
        if (gene_idx + 1) % 5000 == 0:
            print(f'    {gene_idx + 1:,} genes...')

X_geo = sp.csr_matrix(
    (data_list, (rows_list, cols_list)),
    shape=(len(genes), len(idx_cols))
).T
del rows_list, cols_list, data_list

adata_geo = sc.AnnData(
    X=X_geo,
    obs=pd.DataFrame(index=barcodes_sel),
    var=pd.DataFrame(index=genes),
)
adata_geo.var_names_make_unique()
del X_geo, genes

cols_annot = [c for c in ['Sample', 'Cell_type', 'Cell_subtype'] if c in annot.columns]
adata_geo.obs = adata_geo.obs.join(annot[cols_annot], how='left')
adata_geo.obs['batch']     = 'GSE131907_' + adata_geo.obs['Sample'].astype(str)
adata_geo.obs['dataset']   = 'GSE131907'
adata_geo.obs['cell_type'] = adata_geo.obs['Cell_type']
del annot

print(f'  {adata_geo.n_obs:,} células × {adata_geo.n_vars:,} genes')
print('  Tipos celulares:')
for ct, n in adata_geo.obs['cell_type'].value_counts().items():
    print(f'    {ct}: {n}')

Carregando GSE131907 (Kim et al. 2020)...
  Células tumorais selecionadas: 82,595
  Lendo cabeçalho da matriz...
  Carregando 82,595 células (pode levar ~5 min)...
    5,000 genes...
    10,000 genes...
    15,000 genes...
    20,000 genes...
    25,000 genes...
  82,595 células × 29,634 genes
  Tipos celulares:
    T lymphocytes: 43906
    B lymphocytes: 15896
    Myeloid cells: 10082
    Epithelial cells: 7270
    MAST cells: 1809
    Fibroblasts: 1718
    NK cells: 1004
    Endothelial cells: 655
    Undetermined: 255


3. GSE189357 — Ji et al. 2022

só rodou pq corrigi o caminho

In [10]:
samples_gse189357 = {
    'GSM5699777_TD1_': ('TD1', 'AIS'),
    'GSM5699778_TD2_': ('TD2', 'AIS'),
    'GSM5699779_TD3_': ('TD3', 'AIS'),
    'GSM5699780_TD4_': ('TD4', 'MIA'),
    'GSM5699781_TD5_': ('TD5', 'MIA'),
    'GSM5699782_TD6_': ('TD6', 'MIA'),
    'GSM5699783_TD7_': ('TD7', 'IAC'),
    'GSM5699784_TD8_': ('TD8', 'IAC'),
    'GSM5699785_TD9_': ('TD9', 'IAC'),
}

adatas_189357 = []
for prefix, (sample_id, stage) in samples_gse189357.items():
    ad = sc.read_10x_mtx(
        GSE189357_DIR,
        prefix=prefix,
        var_names='gene_symbols',
        make_unique=True,
    )
    ad.obs['sample']    = sample_id
    ad.obs['stage']     = stage
    ad.obs['batch']     = f'GSE189357_{sample_id}'
    ad.obs['dataset']   = 'GSE189357'
    ad.obs['cell_type'] = 'Unknown'
    ad.obs_names        = [f'{sample_id}_{b}' for b in ad.obs_names]
    adatas_189357.append(ad)
    print(f'  {sample_id} ({stage}): {ad.n_obs:,} células')

adata_189357 = sc.concat(adatas_189357, join='outer', merge='same')
adata_189357.var_names_make_unique()
del adatas_189357

print(f'  Total GSE189357: {adata_189357.n_obs:,} células × {adata_189357.n_vars:,} genes')

  TD1 (AIS): 15,216 células
  TD2 (AIS): 18,064 células
  TD3 (AIS): 12,658 células
  TD4 (MIA): 11,756 células
  TD5 (MIA): 19,079 células
  TD6 (MIA): 8,999 células
  TD7 (IAC): 3,507 células
  TD8 (IAC): 15,521 células
  TD9 (IAC): 17,573 células
  Total GSE189357: 122,373 células × 33,538 genes


normalização cada dataset separado na mesma escala

In [11]:
print('QC e normalização...')

def qc_normalize(adata, label):
    print(f'  {label}: {adata.n_obs:,} células antes do QC')
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None,
                                log1p=False, inplace=True)
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_cells(adata, max_genes=6000)
    sc.pp.filter_genes(adata, min_cells=3)
    adata = adata[adata.obs['pct_counts_mt'] < 20].copy()
    print(f'  {label}: {adata.n_obs:,} células após QC')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata

adata_local_raw = qc_normalize(adata_local_raw, 'Local')
adata_geo       = qc_normalize(adata_geo,       'GSE131907')
adata_189357    = qc_normalize(adata_189357,    'GSE189357')



QC e normalização...
  Local: 5,988 células antes do QC
filtered out 122 genes that are detected in less than 3 cells
  Local: 5,988 células após QC
normalizing counts per cell
    finished (0:00:00)
  GSE131907: 82,595 células antes do QC
filtered out 233 cells that have more than 6000 genes expressed
filtered out 4793 genes that are detected in less than 3 cells
  GSE131907: 82,362 células após QC
normalizing counts per cell
    finished (0:00:00)
  GSE189357: 122,373 células antes do QC
filtered out 1344 cells that have less than 200 genes expressed
filtered out 523 cells that have more than 6000 genes expressed
filtered out 6920 genes that are detected in less than 3 cells
  GSE189357: 118,745 células após QC
normalizing counts per cell
    finished (0:00:00)


5. CONCATENAÇÃO APENAS GENES PRESENTES NOS 3 DATASET

In [13]:
print('Concatenando...')

adata_concat = sc.concat(
    [adata_local_raw, adata_geo, adata_189357],
    join='inner',
    label='dataset',
    keys=['local_10x_5GEX', 'GSE131907', 'GSE189357'],
    index_unique='-',
)
del adata_local_raw, adata_geo, adata_189357

print(f'  Total: {adata_concat.n_obs:,} células × {adata_concat.n_vars:,} genes')
print('  Por dataset:')
print(adata_concat.obs['dataset'].value_counts().to_string())


Concatenando...


NameError: name 'adata_local_raw' is not defined